# CE49X Lab 4: Istanbul Earthquake Risk Communication Dashboard
## Visualization for Decision-Makers

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

**Group Members:**
| Name | Student ID |
|------|------------|
| *(your name)* | *(your ID)* |
| *(partner's name)* | *(partner's ID)* |

## Background

Istanbul sits directly on the **North Anatolian Fault (NAF)**, one of the most active strike-slip faults in the world. The devastating 1999 Izmit earthquake (M7.6) ruptured a segment just 80 km east of the city, and seismologists have long warned that the next major rupture is expected beneath the Sea of Marmara — directly south of Istanbul.

The 2023 Kahramanmaras earthquake sequence (M7.8 + M7.5) demonstrated the catastrophic potential of large earthquakes in Turkey and renewed urgency around earthquake preparedness in Istanbul, a city of over 16 million people.

Effective **risk communication** is critical: decision-makers in municipal government need clear, honest, and actionable visualizations — not just raw data. A misleading colormap or a truncated axis can distort risk perception and lead to poor resource allocation.

> **Key Insight:** Visualization is not decoration — it is a critical tool for communicating risk to non-technical audiences. Every design choice (color, scale, annotation) shapes how decision-makers understand and act on data.

## Scenario

You have been hired as a **visualization consultant** for Istanbul's Disaster Coordination Center (AFAD Istanbul). Your task is to create a set of **publication-quality visualizations** for a risk communication report that will be presented to the **Istanbul Municipal Council** — a non-technical audience of elected officials who must decide how to allocate earthquake preparedness funding across Istanbul's districts.

Your visualizations must be:
- **Accurate** — no misleading scales, truncated axes, or rainbow colormaps
- **Clear** — interpretable by someone without a seismology background
- **Actionable** — each chart should support a specific decision or insight
- **Professional** — publication-ready with proper labels, titles, and annotations

## Data Requirements

You must collect **three categories** of real data. No data files are provided — you are responsible for sourcing, downloading, and documenting your data. For each dataset, record the **source URL**, **date accessed**, **how you obtained it**, and **number of records**.

### 1. Earthquake Catalog (required)

Historical earthquakes for the Marmara region. You need **at least 200 earthquakes** with year, latitude, longitude, depth, and magnitude.

Suggested sources:
- **AFAD Earthquake Department**: [deprem.afad.gov.tr](https://deprem.afad.gov.tr) — Turkey's official earthquake catalog, searchable by region/date/magnitude, downloadable as CSV
- **KOERI (Kandilli Observatory)**: [koeri.boun.edu.tr](http://www.koeri.boun.edu.tr) — Bogazici University's own seismology center
- **USGS Earthquake Catalog**: [earthquake.usgs.gov/earthquakes/search](https://earthquake.usgs.gov/earthquakes/search/) — global catalog, CSV export, filter by lat/lon bounding box (e.g., 40-41.5°N, 27-31°E for the Marmara region)

### 2. Building / Population Data (required)

Istanbul district-level data on buildings and/or population. You need data for **at least 10 Istanbul districts** with population and at least one building-related metric (age, type, or count).

Suggested sources:
- **TUIK (Turkish Statistical Institute)**: [data.tuik.gov.tr](https://data.tuik.gov.tr) — district-level population, building permits, construction statistics
- **IBB Open Data Portal**: [data.ibb.gov.tr](https://data.ibb.gov.tr) — Istanbul municipality datasets on buildings, infrastructure, demographics
- **Wikipedia / official district pages** — acceptable for population/area if properly cited

### 3. Seismic Hazard or Vulnerability Data (required)

Any dataset that allows spatial visualization of earthquake risk. You need spatial risk data covering Istanbul (grid, district-level, or at least 10 data points with coordinates).

Options include:
- **AFAD seismic hazard maps** — PGA (Peak Ground Acceleration) values or seismic zones for Istanbul
- **AFAD/IBB building damage estimates** — expected damage scenarios for a major Marmara earthquake
- **Soil classification maps** from IBB or academic papers
- **Any published risk study** with district-level or grid-level data (can be manually digitized from a figure if necessary)

## Deliverables Overview

| # | Title | Points | Key Techniques |
|---|-------|--------|----------------|
| D1 | Data Collection & Documentation | 10 | Data sourcing, Pandas loading, `df.head()`, `df.describe()` |
| D2 | Historical Seismicity Timeline | 15 | Scatter with variable size/color, colorbar, `annotate` |
| D3 | Magnitude-Frequency Analysis | 15 | Histogram + KDE, `axvline`, log-plot with linear fit |
| D4 | Building Vulnerability or Population Risk | 15 | Seaborn categorical plots, sorted charts, color encoding |
| D5 | Earthquake Hazard Visualization | 20 | `contourf`/scatter/heatmap, sequential colormap, annotations |
| D6 | Multi-Panel Risk Dashboard + Reflection | 25 | `GridSpec`, 3+ chart types, derived metric, `savefig`, written reflection |
| **Total** | | **100** | |

---
## Deliverable 1: Data Collection & Documentation (10 pts)

Load all three collected datasets into Pandas DataFrames. For **each** dataset:

1. **Document the source** — URL, date accessed, how you obtained the data (e.g., "downloaded CSV from USGS search interface with bounding box 40-41.5°N, 27-31°E, all magnitudes, 1900-2026")
2. **Show `df.head()`** to display the first few rows
3. **Show `df.describe()`** to summarize key statistics
4. **Note any cleaning steps** performed (dropping NaN rows, converting date formats, renaming columns, etc.)
5. **Report the number of records** in each dataset

> **Key Insight:** Data provenance matters. A visualization is only as trustworthy as the data behind it. Always document your sources so others can verify and reproduce your work.

In [ ]:

import io
import json
import math
import warnings
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300


In [ ]:

def build_usgs_query_url():
    params = {
        "format": "csv",
        "starttime": "1900-01-01",
        "endtime": "2026-03-24",
        "minlatitude": 40.0,
        "maxlatitude": 41.5,
        "minlongitude": 27.0,
        "maxlongitude": 31.0,
        "minmagnitude": 2.0,
        "orderby": "time-asc"
    }
    return "https://earthquake.usgs.gov/fdsnws/event/1/query?" + urlencode(params)


def load_earthquake_catalog():
    url = build_usgs_query_url()
    df = pd.read_csv(url)

    df = df.rename(columns={
        "time": "time_utc",
        "latitude": "latitude",
        "longitude": "longitude",
        "depth": "depth_km",
        "mag": "magnitude",
        "place": "place"
    })

    keep_cols = ["time_utc", "latitude", "longitude", "depth_km", "magnitude", "place", "type", "magType"]
    df = df[keep_cols].copy()

    df["time_utc"] = pd.to_datetime(df["time_utc"], errors="coerce")
    df["year"] = df["time_utc"].dt.year

    df = df.dropna(subset=["time_utc", "latitude", "longitude", "depth_km", "magnitude"]).reset_index(drop=True)

    return df, url


def arcgis_query_json(url, params):
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def load_district_scenario_attributes():
    base_url = (
        "https://services-eu1.arcgis.com/LHwUjP01iDaGy6Hk/ArcGIS/rest/services/"
        "%C4%B0stanbul_%C4%B0l%C3%A7elere_G%C3%B6re_Deprem_Hasar_Tahmini/FeatureServer/1/query"
    )

    params = {
        "where": "1=1",
        "outFields": ",".join([
            "Adi",
            "ToplamNufus2022",
            "cok_agir_hasarli_bina_sayisi",
            "agir_hasarli_bina_sayisi",
            "orta_hasarli_bina_sayisi",
            "hafif_hasarli_bina_sayisi",
            "cokveagir_hasarli_bina_orani",
            "can_kaybi_sayisi",
            "agir_yarali_sayisi",
            "hastanede_tedavi_sayisi",
            "hafif_yarali_sayisi",
            "gecici_barinma"
        ]),
        "returnGeometry": "false",
        "f": "json"
    }

    data = arcgis_query_json(base_url, params)
    rows = [feature["attributes"] for feature in data["features"]]
    df = pd.DataFrame(rows)

    df = df.rename(columns={
        "Adi": "district",
        "ToplamNufus2022": "population_2022",
        "cok_agir_hasarli_bina_sayisi": "very_heavy_damage_buildings",
        "agir_hasarli_bina_sayisi": "heavy_damage_buildings",
        "orta_hasarli_bina_sayisi": "moderate_damage_buildings",
        "hafif_hasarli_bina_sayisi": "light_damage_buildings",
        "cokveagir_hasarli_bina_orani": "severe_damage_rate_pct",
        "can_kaybi_sayisi": "fatalities_est",
        "agir_yarali_sayisi": "serious_injuries_est",
        "hastanede_tedavi_sayisi": "hospital_treatment_est",
        "hafif_yarali_sayisi": "minor_injuries_est",
        "gecici_barinma": "temporary_shelter_need"
    })

    numeric_cols = [c for c in df.columns if c != "district"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    return df, base_url


def load_district_scenario_geojson():
    url = (
        "https://services-eu1.arcgis.com/LHwUjP01iDaGy6Hk/ArcGIS/rest/services/"
        "%C4%B0stanbul_%C4%B0l%C3%A7elere_G%C3%B6re_Deprem_Hasar_Tahmini/FeatureServer/1/query"
    )

    params = {
        "where": "1=1",
        "outFields": ",".join([
            "Adi",
            "ToplamNufus2022",
            "cok_agir_hasarli_bina_sayisi",
            "agir_hasarli_bina_sayisi",
            "cokveagir_hasarli_bina_orani",
            "can_kaybi_sayisi",
            "gecici_barinma"
        ]),
        "returnGeometry": "true",
        "f": "geojson"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    geojson_data = response.json()
    return geojson_data, response.url


def simple_centroid_from_geometry(geometry):
    '''
    Straightforward centroid approximation.
    This is not a full GIS-grade centroid calculation, but it is good enough
    for a district-center bubble map used in a communication-focused lab.
    '''
    coords = []

    if geometry["type"] == "Polygon":
        for ring in geometry["coordinates"]:
            coords.extend(ring)

    elif geometry["type"] == "MultiPolygon":
        for polygon in geometry["coordinates"]:
            for ring in polygon:
                coords.extend(ring)

    else:
        return np.nan, np.nan

    arr = np.array(coords)
    lon = arr[:, 0].mean()
    lat = arr[:, 1].mean()
    return lon, lat


def build_spatial_risk_table(geojson_data):
    rows = []

    for feature in geojson_data["features"]:
        props = feature["properties"]
        lon, lat = simple_centroid_from_geometry(feature["geometry"])

        row = {
            "district": props["Adi"],
            "population_2022": props["ToplamNufus2022"],
            "very_heavy_damage_buildings": props["cok_agir_hasarli_bina_sayisi"],
            "heavy_damage_buildings": props["agir_hasarli_bina_sayisi"],
            "severe_damage_rate_pct": props["cokveagir_hasarli_bina_orani"],
            "fatalities_est": props["can_kaybi_sayisi"],
            "temporary_shelter_need": props["gecici_barinma"],
            "lon": lon,
            "lat": lat
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    numeric_cols = [c for c in df.columns if c != "district"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df


def load_district_area_table():
    wiki_url = "https://en.wikipedia.org/wiki/List_of_districts_of_Istanbul"
    tables = pd.read_html(wiki_url)

    # Find the table that contains District / Area / Population
    selected = None
    for table in tables:
        cols = [str(c) for c in table.columns]
        if any("District" in c for c in cols) and any("Area" in c for c in cols):
            selected = table.copy()
            break

    if selected is None:
        raise ValueError("Could not find the Istanbul district area table on Wikipedia.")

    # Flatten columns if the table has multi-level columns
    if isinstance(selected.columns, pd.MultiIndex):
        selected.columns = [" ".join([str(x) for x in col if str(x) != "nan"]).strip() for col in selected.columns]

    selected.columns = [str(c).strip() for c in selected.columns]

    district_col = [c for c in selected.columns if "District" in c][0]
    area_col = [c for c in selected.columns if "Area" in c][0]

    area_df = selected[[district_col, area_col]].copy()
    area_df.columns = ["district", "area_km2"]
    area_df["district"] = area_df["district"].astype(str).str.strip()
    area_df["area_km2"] = pd.to_numeric(area_df["area_km2"], errors="coerce")

    return area_df, wiki_url


def build_population_building_table(district_df, area_df):
    merged = district_df.merge(area_df, on="district", how="left")
    merged["population_density"] = merged["population_2022"] / merged["area_km2"]
    merged["severe_damage_buildings"] = (
        merged["very_heavy_damage_buildings"] + merged["heavy_damage_buildings"]
    )
    merged["fatalities_per_100k"] = 100000 * merged["fatalities_est"] / merged["population_2022"]
    merged["shelter_need_pct"] = 100 * merged["temporary_shelter_need"] / merged["population_2022"]
    merged["density_x_damage_rate"] = merged["population_density"] * merged["severe_damage_rate_pct"]
    return merged


def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)


def add_derived_risk_index(df):
    out = df.copy()
    out["risk_index"] = (
        0.45 * zscore(out["severe_damage_rate_pct"]) +
        0.30 * zscore(out["fatalities_per_100k"]) +
        0.25 * zscore(out["shelter_need_pct"])
    )
    return out.sort_values("risk_index", ascending=False).reset_index(drop=True)


In [ ]:

earthquake_df, usgs_url = load_earthquake_catalog()
district_attr_df, arcgis_attr_url = load_district_scenario_attributes()
district_geojson, arcgis_geojson_url = load_district_scenario_geojson()
district_area_df, wiki_url = load_district_area_table()

population_building_df = build_population_building_table(district_attr_df, district_area_df)
hazard_df = build_spatial_risk_table(district_geojson)
hazard_df = hazard_df.merge(
    population_building_df[["district", "population_density", "fatalities_per_100k", "shelter_need_pct"]],
    on="district",
    how="left"
)
hazard_df = add_derived_risk_index(hazard_df)

print("Earthquake records:", len(earthquake_df))
print("District exposure/building records:", len(population_building_df))
print("Spatial hazard/vulnerability records:", len(hazard_df))


print("USGS query URL:")
print(usgs_url)
print()
print("ArcGIS attributes URL:")
print(arcgis_attr_url)
print()
print("ArcGIS geojson URL:")
print(arcgis_geojson_url)
print()
print("Wikipedia URL:")
print(wiki_url)


print("Dataset 1 - Earthquake catalog")
print("Number of records:", len(earthquake_df))
display(earthquake_df.head())
display(earthquake_df.describe(include="all"))


print("Dataset 2 - District exposure / building vulnerability")
print("Number of records:", len(population_building_df))
display(population_building_df.head())
display(population_building_df.describe(include="all"))


print("Dataset 3 - Spatial hazard / vulnerability table")
print("Number of records:", len(hazard_df))
display(hazard_df.head())
display(hazard_df.describe(include="all"))


---
## Deliverable 2: Historical Seismicity Timeline

In [ ]:

# Count larger earthquakes for a story-driven title
num_m6 = int((earthquake_df["magnitude"] >= 6.0).sum())

# Try to find 3 major earthquakes for annotation from the catalog itself.
annotation_candidates = [
    ("1999 Izmit", "Izmit"),
    ("1999 Duzce", "Duzce"),
    ("2025 Marmara Ereglisi", "Marmara Ereğlisi"),
]

found_annotations = []
for label, keyword in annotation_candidates:
    matches = earthquake_df[earthquake_df["place"].str.contains(keyword, case=False, na=False)].copy()
    if not matches.empty:
        # choose the largest matching event
        best = matches.sort_values("magnitude", ascending=False).iloc[0]
        found_annotations.append((label, best))

fig, ax = plt.subplots(figsize=(15, 7))

sizes = 10 + (earthquake_df["magnitude"] ** 3) * 2.5
sc = ax.scatter(
    earthquake_df["time_utc"],
    earthquake_df["magnitude"],
    s=sizes,
    c=earthquake_df["depth_km"],
    cmap="viridis",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.3
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Depth (km)")

for label, row in found_annotations:
    ax.annotate(
        f"{label}\nM{row['magnitude']:.1f}",
        xy=(row["time_utc"], row["magnitude"]),
        xytext=(10, 12),
        textcoords="offset points",
        fontsize=10,
        arrowprops=dict(arrowstyle="->", lw=1.0)
    )

ax.set_title(
    f"Marmara earthquake catalog shows {num_m6} earthquakes above M6.0 in the study window",
    pad=15,
    fontsize=18,
    weight="bold"
)
ax.set_xlabel("Date")
ax.set_ylabel("Magnitude")
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


---
## Deliverable 3: Magnitude-Frequency Analysis

In [ ]:

magnitudes = earthquake_df["magnitude"].dropna().values
mean_mag = np.mean(magnitudes)
median_mag = np.median(magnitudes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Using 0.2 magnitude bins because the catalog commonly reports magnitudes
# to around 0.1 precision, and 0.2 keeps the histogram readable without
# creating unnecessary visual noise.
bins = np.arange(np.floor(magnitudes.min()), np.ceil(magnitudes.max()) + 0.2, 0.2)

sns.histplot(magnitudes, bins=bins, kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(mean_mag, color="darkred", linestyle="--", linewidth=2, label=f"Mean = {mean_mag:.2f}")
axes[0].axvline(median_mag, color="black", linestyle=":", linewidth=2, label=f"Median = {median_mag:.2f}")
axes[0].set_title("Most recorded earthquakes are small, with a right tail of larger events")
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Count")
axes[0].legend()

# Gutenberg-Richter plot
mags = np.sort(magnitudes)
unique_mags = np.arange(np.floor(mags.min() * 10) / 10, np.ceil(mags.max() * 10) / 10 + 0.1, 0.1)
cumulative_counts = np.array([np.sum(mags >= m) for m in unique_mags])

mask = cumulative_counts > 0
gr_df = pd.DataFrame({
    "magnitude": unique_mags[mask],
    "count_geq_M": cumulative_counts[mask]
})

gr_df["log10_count"] = np.log10(gr_df["count_geq_M"])

# Use only the more linear mid-range section for the fit.
fit_mask = (gr_df["magnitude"] >= 3.0) & (gr_df["magnitude"] <= 5.5)
fit_x = gr_df.loc[fit_mask, "magnitude"]
fit_y = gr_df.loc[fit_mask, "log10_count"]

coeffs = np.polyfit(fit_x, fit_y, 1)
slope, intercept = coeffs
b_value = -slope

axes[1].scatter(gr_df["magnitude"], gr_df["log10_count"], s=35, alpha=0.8)
axes[1].plot(
    fit_x,
    slope * fit_x + intercept,
    linestyle="--",
    linewidth=2,
    label=f"Linear fit (b = {b_value:.2f})"
)
axes[1].set_title("Log transformation reveals an approximately linear magnitude-frequency relation")
axes[1].set_xlabel("Magnitude")
axes[1].set_ylabel("log$_{10}$(Cumulative count ≥ M)")
axes[1].legend()

plt.tight_layout()
plt.show()


---
## Deliverable 4: Building Vulnerability or Population Risk

In [ ]:

top_density = population_building_df.sort_values("population_density", ascending=False).head(15).copy()
top_severe_rate = population_building_df.sort_values("severe_damage_rate_pct", ascending=False).head(15).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Seaborn categorical plot
palette_1 = ["#c44e52" if i < 5 else "#4c72b0" for i in range(len(top_density))]
sns.barplot(
    data=top_density,
    x="district",
    y="population_density",
    palette=palette_1,
    ax=axes[0]
)
axes[0].set_title("High-density districts concentrate more people in limited space")
axes[0].set_xlabel("")
axes[0].set_ylabel("Population density (people per km$^2$)")
axes[0].tick_params(axis="x", rotation=65)

palette_2 = ["#dd8452" if i < 5 else "#55a868" for i in range(len(top_severe_rate))]
sns.barplot(
    data=top_severe_rate,
    x="district",
    y="severe_damage_rate_pct",
    palette=palette_2,
    ax=axes[1]
)
axes[1].set_title("Several districts stand out for severe building damage rate in the scenario")
axes[1].set_xlabel("")
axes[1].set_ylabel("Very heavy + heavy damage rate (%)")
axes[1].tick_params(axis="x", rotation=65)

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(10, 7))

scatter_df = population_building_df.copy()

ax.scatter(
    scatter_df["population_density"],
    scatter_df["severe_damage_buildings"],
    s=50 + scatter_df["fatalities_est"] / 2,
    alpha=0.75
)

for _, row in scatter_df.nlargest(12, "severe_damage_buildings").iterrows():
    ax.annotate(
        row["district"],
        (row["population_density"], row["severe_damage_buildings"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.set_title("Districts with both dense population and high building damage deserve close attention")
ax.set_xlabel("Population density (people per km$^2$)")
ax.set_ylabel("Severe damage buildings (very heavy + heavy)")
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


---
## Deliverable 5: Earthquake Hazard Visualization

In [ ]:

map_df = hazard_df.copy()
map_df["severe_damage_buildings"] = (
    map_df["very_heavy_damage_buildings"] + map_df["heavy_damage_buildings"]
)

top_risk = map_df.nlargest(6, "risk_index")

fig, ax = plt.subplots(figsize=(11, 9))

sc = ax.scatter(
    map_df["lon"],
    map_df["lat"],
    c=map_df["risk_index"],
    s=40 + map_df["severe_damage_buildings"] / 20,
    cmap="YlOrRd",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.4
)

for _, row in map_df.iterrows():
    ax.text(row["lon"], row["lat"], row["district"], fontsize=8, ha="center", va="center")

for _, row in top_risk.iterrows():
    ax.annotate(
        f"{row['district']}",
        xy=(row["lon"], row["lat"]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-", lw=1)
    )

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Derived district risk index")

ax.text(28.75, 40.83, "Sea of Marmara", fontsize=12, style="italic", alpha=0.8)
ax.text(29.10, 41.10, "Bosphorus", fontsize=11, rotation=65, alpha=0.8)

ax.set_title("Scenario risk clusters along several Marmara-facing districts", pad=12, fontsize=17, weight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


---
## Deliverable 6: Multi-Panel Risk Dashboard + Reflection

In [ ]:

dashboard_df = add_derived_risk_index(population_building_df.copy())
dashboard_top10 = dashboard_df.nlargest(10, "risk_index").sort_values("risk_index", ascending=True)

fig = plt.figure(figsize=(18, 11))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.30)

ax1 = fig.add_subplot(gs[0, :2])   # spans 2 columns
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1:])   # spans 2 columns

# Panel 1 - timeline
sizes = 8 + (earthquake_df["magnitude"] ** 3) * 2
sc1 = ax1.scatter(
    earthquake_df["time_utc"],
    earthquake_df["magnitude"],
    s=sizes,
    c=earthquake_df["depth_km"],
    cmap="viridis",
    alpha=0.7,
    edgecolor="white",
    linewidth=0.2
)
ax1.set_title("Historical seismicity")
ax1.set_xlabel("Date")
ax1.set_ylabel("Magnitude")
ax1.grid(True, alpha=0.2)

# Panel 2 - histogram
sns.histplot(earthquake_df["magnitude"], bins=bins, kde=True, ax=ax2, color="slateblue")
ax2.axvline(mean_mag, color="darkred", linestyle="--", linewidth=2)
ax2.axvline(median_mag, color="black", linestyle=":", linewidth=2)
ax2.set_title("Magnitude distribution")
ax2.set_xlabel("Magnitude")
ax2.set_ylabel("Count")

# Panel 3 - top risk districts
ax3.barh(dashboard_top10["district"], dashboard_top10["risk_index"])
ax3.set_title("Top 10 districts by derived risk index")
ax3.set_xlabel("Risk index")
ax3.set_ylabel("District")

# Panel 4 - spatial risk map
map_df = hazard_df.copy()
map_df["severe_damage_buildings"] = map_df["very_heavy_damage_buildings"] + map_df["heavy_damage_buildings"]
sc4 = ax4.scatter(
    map_df["lon"],
    map_df["lat"],
    c=map_df["risk_index"],
    s=40 + map_df["severe_damage_buildings"] / 20,
    cmap="YlOrRd",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.4
)
for _, row in map_df.nlargest(10, "risk_index").iterrows():
    ax4.text(row["lon"], row["lat"], row["district"], fontsize=8, ha="center", va="center")

ax4.text(28.75, 40.83, "Sea of Marmara", fontsize=12, style="italic", alpha=0.8)
ax4.text(29.10, 41.10, "Bosphorus", fontsize=11, rotation=65, alpha=0.8)
ax4.set_title("District-level spatial risk")
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")
ax4.set_aspect("equal", adjustable="box")
ax4.grid(True, alpha=0.2)

fig.suptitle(
    "Istanbul earthquake preparedness should prioritize districts where dense population overlaps with high projected damage",
    fontsize=20,
    weight="bold",
    y=1.02
)

fig.savefig("dashboard.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()



## Written Reflection

### 1) Chart type justification

**D2 - Historical seismicity timeline**  
I used a **scatter plot** because the most important variables are **time** and **magnitude**, and position is the most accurate visual encoding. This matches the **encoding hierarchy** idea from visualization theory. Marker size adds emphasis for larger events, while depth is shown with a sequential colormap and colorbar. This allows one chart to communicate three variables without hiding the main story.

**D3 - Magnitude-frequency analysis**  
For the first panel I used a **histogram with KDE** because the task is to show the shape of the magnitude distribution. This follows the **chart selection framework**: a histogram is appropriate for one quantitative variable. For the second panel I used a **Gutenberg-Richter scatter plot in log space** because the log transform makes the magnitude-frequency relationship visible. This directly applies the theory idea that a good transformation can reveal a pattern that is hard to see in raw scale.

**D4 - District comparison charts**  
I used **sorted bar charts** because district names are categorical and the goal is comparison. Sorting is important because alphabetical order would weaken the message. The first bar chart compares population density, and the second compares severe damage rate. This follows both the **comparison chart rule** and **data-ink ratio** principle: the chart should make ranking obvious without decorative clutter.

**D5 - Spatial hazard visualization**  
I used a **district-level bubble map** because the problem is spatial and the council must know **where** the greatest risk is. A sequential colormap (`YlOrRd`) was used instead of rainbow colors because theory warns that rainbow maps create false boundaries and confuse interpretation. I also added annotations and geographic context labels so the audience can connect the pattern to real places.

---

### 2) Audience adaptation: what would change for seismologists?

If the audience were **seismologists** instead of the municipal council, I would make at least two important changes:

**First**, I would replace some simplified communication choices with more technical measures.  
For example, instead of a derived risk index, I would show the underlying variables separately such as **PGA**, **site amplification**, **catalog completeness threshold**, and fit sensitivity for the Gutenberg-Richter relation.

**Second**, I would increase technical detail and uncertainty reporting.  
For example, I would show **fit residuals**, the chosen **magnitude completeness level**, and maybe confidence intervals for the regression slope. For the district risk layer, I would also state more clearly that the scenario results depend on model assumptions and are not deterministic forecasts.

---

### 3) Self-critique

My weakest visualization is the **district-level bubble map**.  
It is useful for communication, but it is still an approximation because I am placing district labels at simple centroid locations rather than using a full choropleth or a more advanced GIS-based centroid method. The main theory-related weakness is potential **over-plotting** and mild label crowding in the central districts.

To improve it, I would:
- use a true district polygon map,
- reduce overlapping labels with smarter annotation placement,
- and possibly split the view into European and Asian sides.

That would reduce clutter and improve spatial precision without adding chartjunk.
